# 🎮 Video Games Sentiment Analysis (Analisis Sentimen Ulasan Video Games)

Notebook ini mengimplementasikan alur lengkap analisis sentimen berbasis **Natural Language Processing (NLP)** dan **Machine Learning** pada dataset **Amazon Reviews 2023** kategori **Video_Games**.

### Alur Proyek:
1. **Persiapan Lingkungan & Pustaka**
2. **Pengunduhan & Pemuatan Data Bertahap (Streaming / Chunking)**
   - Membaca data `.jsonl.gz` baris demi baris menggunakan `json.loads` agar memori RAM tidak penuh.
   - Memfilter ulasan selama pembacaan: hanya ulasan dengan `timestamp <= 1577836799999` (hingga 31 Desember 2019 UTC).
3. **Pembersihan Data & Pelabelan (Preprocessing)**
   - Membuang baris dengan `rating == 3` atau nilai kosong pada `rating`, `title`, dan `text`.
   - Membuat label biner: rating 1 atau 2 $\rightarrow$ `0` (Negatif), rating 4 atau 5 $\rightarrow$ `1` (Positif).
   - Menggabungkan kolom `title` dan `text` menjadi `full_text`.
   - Melakukan pembersihan dasar (whitespace berlebih & karakter tidak relevan).
4. **Exploratory Data Analysis (EDA)**
5. **Pembagian Data (Train & Test Split)**
6. **Ekstraksi Fitur & Pipeline Pemodelan (TF-IDF + Logistic Regression)**
7. **Evaluasi Model** (Accuracy, Precision, Recall, F1-Score & Confusion Matrix)
8. **Penyimpanan Model Terlatih** (`model/model_sentimen.joblib`)
9. **Pengujian Inferensi Real-time**

## 1. Persiapan Lingkungan & Pemuatan Pustaka

In [1]:
import os
import re
import gzip
import json
import zlib
import time
import datetime
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")
print("Pustaka berhasil dimuat!")

Pustaka berhasil dimuat!


## 2. Pengunduhan & Pemuatan Data secara Bertahap (Chunking/Streaming)

> **Penting:** Dataset Amazon Reviews 2023 kategori `Video_Games` berukuran besar (arsip `.jsonl.gz` terkompresi ~814 MB dan beberapa gigabyte saat diekstrak). Untuk mencegah kehabisan memori RAM, kita membaca berkas secara **streaming baris demi baris (line-by-line)** menggunakan `json.loads`.
>
> **Aturan Filter:** Hanya ulasan dengan `timestamp <= 1577836799999` (31 Desember 2019, 23:59:59.999 UTC) yang diambil.

In [2]:
URL_VIDEO_GAMES = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz"
CUTOFF_TIMESTAMP = 1577836799999  # 31 Desember 2019 23:59:59.999 UTC

def stream_and_filter_reviews(url=URL_VIDEO_GAMES, max_samples=5000, cutoff_timestamp=CUTOFF_TIMESTAMP):
    """
    Membaca berkas .jsonl.gz secara streaming lewat HTTP chunks,
    mendekompresi on-the-fly, membaca baris demi baris dengan json.loads,
    serta memfilter data ulasan dengan timestamp <= cutoff_timestamp.
    """
    print(f"Mulai streaming dari: {url}")
    print(f"Filter batas timestamp: <= {cutoff_timestamp} (Hingga 31 Des 2019)")
    
    start_time = time.time()
    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    
    d = zlib.decompressobj(16 + zlib.MAX_WBITS)
    records = []
    buffer = ""
    total_scanned = 0
    
    for chunk in resp.iter_content(chunk_size=65536):
        if not chunk:
            continue
        decompressed = d.decompress(chunk)
        buffer += decompressed.decode("utf-8", errors="ignore")
        lines = buffer.split("\n")
        buffer = lines[-1]
        
        for line in lines[:-1]:
            line_str = line.strip()
            if not line_str:
                continue
            total_scanned += 1
            record = json.loads(line_str)
            ts = record.get("timestamp")
            
            # Filter ulasan: timestamp <= 31 Des 2019
            if ts is not None and ts <= cutoff_timestamp:
                text = record.get("text", "")
                if text and len(text.strip()) > 5:
                    rating = float(record.get("rating", 0))
                    dt = datetime.datetime.fromtimestamp(ts / 1000.0, datetime.timezone.utc)
                    records.append({
                        "rating": rating,
                        "title": record.get("title", ""),
                        "text": text.strip(),
                        "timestamp": ts,
                        "date": dt.strftime("%Y-%m-%d %H:%M:%S"),
                        "year": dt.year,
                        "asin": record.get("asin", record.get("parent_asin", "")),
                        "verified_purchase": record.get("verified_purchase", False),
                        "helpful_vote": record.get("helpful_vote", 0)
                    })
            
            if len(records) >= max_samples:
                break
        if len(records) >= max_samples:
            break
            
    elapsed = time.time() - start_time
    print(f"Selesai! Memindai {total_scanned} baris, mendapatkan {len(records)} ulasan tersaring dalam {elapsed:.2f} detik.")
    return pd.DataFrame(records)

In [3]:
# Pemuatan ulasan tersaring dari berkas lokal atau streaming langsung
raw_csv_path = os.path.join("data", "video_games_reviews_2019.csv")

if os.path.exists(raw_csv_path):
    print(f"Memuat data ulasan dari berkas lokal: {raw_csv_path}")
    df = pd.read_csv(raw_csv_path)
else:
    df = stream_and_filter_reviews(max_samples=5000)
    os.makedirs("data", exist_ok=True)
    df.to_csv(raw_csv_path, index=False)

print(f"Total baris data awal: {len(df)}")
print(f"Maksimum timestamp   : {df['timestamp'].max()} (Batas: {CUTOFF_TIMESTAMP})")
print(f"Rentang tanggal      : {df['date'].min()} s/d {df['date'].max()}")
print("Verifikasi <= 2019   :", (df['timestamp'] <= CUTOFF_TIMESTAMP).all())
df.head()

## 3. Pembersihan Data & Pelabelan (Preprocessing)

Sesuai instruksi pemrosesan:
1. **Buang baris** yang memiliki `rating == 3` (ulasan netral) atau nilai kosong (`NaN` / string kosong) pada `rating`, `title`, dan `text`.
2. **Buat label biner (`label`)**:
   - `rating` 1 atau 2 $\rightarrow$ **`0` (Negatif)**
   - `rating` 4 atau 5 $\rightarrow$ **`1` (Positif)**
3. **Gabungkan kolom `title` dan `text`** menjadi satu fitur teks utama:
   $$df[\text{'full\_text'}] = df[\text{'title'}] + \text{" "} + df[\text{'text'}]$$
4. **Pembersihan dasar**: Menghapus tag HTML, URL, karakter aneh/tidak relevan, serta spasi/whitespace berlebih.

In [4]:
# 1. Tangani nilai kosong pada rating, title, dan text
print(f"Jumlah baris sebelum pembersihan: {len(df)}")

# Konversi string kosong / hanya spasi menjadi NaN agar terdeteksi dropna
df["title"] = df["title"].replace(r"^\s*$", np.nan, regex=True)
df["text"] = df["text"].replace(r"^\s*$", np.nan, regex=True)

# Buang baris dengan nilai kosong pada rating, title, atau text
df_clean = df.dropna(subset=["rating", "title", "text"]).copy()

# Buang baris dengan rating == 3 (ulasan netral)
df_clean = df_clean[df_clean["rating"] != 3].copy()
print(f"Jumlah baris setelah membuang rating == 3 dan nilai kosong: {len(df_clean)}")

# 2. Buat kolom target/label biner:
# rating 1 atau 2 -> 0 (Negatif)
# rating 4 atau 5 -> 1 (Positif)
df_clean["label"] = df_clean["rating"].apply(
    lambda r: 0 if r in [1, 2, 1.0, 2.0] else (1 if r in [4, 5, 4.0, 5.0] else np.nan)
)
df_clean = df_clean.dropna(subset=["label"]).copy()
df_clean["label"] = df_clean["label"].astype(int)

# 3. Gabungkan kolom title dan text menjadi satu fitur teks utama
df_clean["full_text"] = df_clean["title"].astype(str).str.strip() + " " + df_clean["text"].astype(str).str.strip()

# 4. Pembersihan dasar (whitespace berlebih & karakter tidak relevan)
def basic_clean_text(text: str) -> str:
    """
    Membersihkan teks ulasan dari tag HTML, URL, karakter tidak relevan,
    dan merapikan spasi berlebih.
    """
    # Hapus tag HTML
    text = re.sub(r"<.*?>", " ", text)
    # Hapus URL
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    # Hapus karakter tidak relevan/simbol aneh (pertahankan alfanumerik & tanda baca dasar)
    text = re.sub(r"[^\w\s.,!?'\"-]", " ", text)
    # Hapus whitespace / spasi berlebih
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_clean["clean_text"] = df_clean["full_text"].apply(basic_clean_text)

# Simpan dataset hasil pra-pemrosesan
preprocessed_csv_path = os.path.join("data", "video_games_preprocessed.csv")
df_clean.to_csv(preprocessed_csv_path, index=False)
print(f"Dataset hasil preprocessing tersimpan di: {preprocessed_csv_path}")

In [5]:
# Tinjau distribusi label biner dan perbandingan teks sebelum vs sesudah
print("Distribusi Label Sentimen Biner:")
counts = df_clean["label"].value_counts()
for lbl, count in counts.items():
    nama = "Positif (1)" if lbl == 1 else "Negatif (0)"
    persen = (count / len(df_clean)) * 100
    print(f"  {nama}: {count} sampel ({persen:.2f}%)")

print("\n--- Contoh Perbandingan Fitur Teks ---")
for i in [0, 1]:
    sample = df_clean.iloc[i]
    print(f"\n[Rating {sample['rating']} -> Label {sample['label']}]")
    print(f"Title      : {sample['title']}")
    print(f"Full Text  : {sample['full_text'][:110]}...")
    print(f"Clean Text : {sample['clean_text'][:110]}...")

## 4. Exploratory Data Analysis (EDA)
Menganalisis visualisasi distribusi rating, perbandingan label sentimen biner, dan panjang kata ulasan.

In [6]:
df_clean["word_count"] = df_clean["clean_text"].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribusi Rating Asli
sns.countplot(data=df_clean, x="rating", ax=axes[0], palette="mako")
axes[0].set_title("Distribusi Rating Ulasan (Tanpa Rating 3)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Jumlah Ulasan")

# 2. Distribusi Label Biner
sns.countplot(data=df_clean, x="label", ax=axes[1], palette=["#e74c3c", "#2ecc71"])
axes[1].set_xticklabels(["Negatif (0)", "Positif (1)"])
axes[1].set_title("Distribusi Label Target Biner", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Jumlah Ulasan")

# 3. Distribusi Panjang Kata
sns.histplot(data=df_clean, x="word_count", hue="label", kde=True, ax=axes[2], palette=["#e74c3c", "#2ecc71"], bins=30)
axes[2].set_xlim(0, 150)
axes[2].set_title("Distribusi Jumlah Kata per Ulasan", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Jumlah Kata")

plt.tight_layout()
plt.show()

## 5. Pembagian Data (Train & Test Split)
Membagi fitur `clean_text` dan target `label` menjadi 80% data latih (training) dan 20% data uji (testing) secara terstratifikasi.

In [7]:
X = df_clean["clean_text"]
y = df_clean["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data Latih (Train): {len(X_train)} sampel")
print(f"Data Uji (Test)   : {len(X_test)} sampel")

## 6. Pipeline Pemodelan (TF-IDF + Logistic Regression)
Menggunakan `scikit-learn` Pipeline: ekstraksi fitur teks dengan `TfidfVectorizer` (unigram & bigram, sublinear TF) dan algoritma klasifikasi `LogisticRegression` berbobot seimbang (`class_weight='balanced'`).

In [8]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        stop_words="english",
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Pelatihan model
pipeline.fit(X_train, y_train)
print("Model Pipeline berhasil dilatih!")

## 7. Evaluasi Model
Menghitung metrik Akurasi, Precision, Recall, F1-Score, dan visualisasi Confusion Matrix Heatmap.

In [9]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Akurasi Model: {acc * 100:.2f}%")
print("\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred, target_names=["Negatif (0)", "Positif (1)"]))

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Negatif (0)", "Pred Positif (1)"],
            yticklabels=["Aktual Negatif (0)", "Aktual Positif (1)"])
plt.title("Confusion Matrix Evaluasi Model Sentimen", fontsize=12, fontweight="bold")
plt.ylabel("Kelas Aktual")
plt.xlabel("Kelas Prediksi")
plt.tight_layout()
plt.show()

## 8. Penyimpanan Model (Model Serialization)
Menyimpan model pipeline ke folder `model/model_sentimen.joblib`.

In [10]:
os.makedirs("model", exist_ok=True)
model_path = os.path.join("model", "model_sentimen.joblib")
joblib.dump(pipeline, model_path)
# Salin juga ke root direktori proyek
joblib.dump(pipeline, "model_sentimen.joblib")
print(f"Model berhasil disimpan di: {model_path} dan model_sentimen.joblib")

## 9. Pengujian Inferensi Real-time (Uji Ulasan Baru)
Menguji model yang tersimpan dengan contoh ulasan baru (kombinasi judul dan isi ulasan game).

In [11]:
loaded_model = joblib.load(model_path)

contoh_ulasan_baru = [
    "Masterpiece of RPG! The combat is smooth, story is deep, and visuals are stunning.",
    "Broken launch and terrible performance. Constant crashing and game breaking bugs.",
    "Great co-op fun with friends. Excellent soundtrack and fun boss fights.",
    "Greedy pay to win microtransactions. Boring repetitive gameplay and empty lobbies."
]

prediksi = loaded_model.predict(contoh_ulasan_baru)
probabilitas = loaded_model.predict_proba(contoh_ulasan_baru)

for teks, pred, prob in zip(contoh_ulasan_baru, prediksi, probabilitas):
    label = "POSITIF" if pred == 1 else "NEGATIF"
    conf = max(prob) * 100
    print(f"Ulasan    : \"{teks}\"")
    print(f"Sentimen  : {label} (Keyakinan: {conf:.2f}%)")
    print("-" * 70)